In [ ]:

import os
RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
print("results/ exists:", os.path.exists(RESULTS_DIR))
print("Files found:", len([f for f in os.listdir(RESULTS_DIR) if f.endswith('.csv')]))

In [ ]:
# ============================================================
# 11_Final_Comparison.ipynb
# Unified comparison table and figure.
# Tabular models use the 65-feature summary of the full observed
# history; sequence models use the full observed sequence. All numbers
# are computed from the saved prediction files and seed CSVs, so the
# table cannot drift from the experiments that produced it.
# Stochastic models are reported as the mean over five seeds.
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
FIGURES_DIR = os.path.join(DATA_DIR, "..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

# --- Per-engine predictions: every model sees the full observed history ---
pred_files = {
    "LR":        ("fd001_lr_full_history.csv",      "pred_LR_full",      "features"),
    "XGBoost":   ("fd001_xgb_full_history.csv",     "pred_XGB_full",     "features"),
    "LSTM":      ("fd001_lstm_full_history.csv",    "pred_LSTM_full",    "sequence"),
    "Chronos-2": ("fd001_chronos_full_history.csv", "pred_Chronos_full", "sequence"),
}

rows = []
for name, (fn, col, inp) in pred_files.items():
    df = pd.read_csv(os.path.join(RESULTS_DIR, fn))
    yt, yp = df["RUL_true"].values, df[col].values
    rows.append({"Model": name, "Input": inp,
                 "MAE_file":  mean_absolute_error(yt, yp),
                 "RMSE_file": np.sqrt(mean_squared_error(yt, yp)),
                 "NASA_file": nasa_score(yt, yp)})
results = pd.DataFrame(rows)

# --- Multi-seed means for the stochastic models ---
seed_files = {
    "XGBoost":   "fd001_multiseed_xgb65.csv",
    "LSTM":      "fd001_multiseed_lstm_full.csv",
    "Chronos-2": "fd001_multiseed_chronos_full.csv",
}
seed_info = {}
for name, fn in seed_files.items():
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.exists(p):
        s = pd.read_csv(p)
        seed_info[name] = {"MAE_mean": s["mae"].mean(),  "MAE_std": s["mae"].std(),
                           "RMSE_mean": s["rmse"].mean(), "NASA_mean": s["nasa"].mean(),
                           "n_seeds": len(s)}
    else:
        print(f"WARNING: {fn} not found; {name} reported from its prediction file only")

def rep(name, metric):
    if name in seed_info:
        return seed_info[name][f"{metric}_mean"]
    return results.loc[results.Model == name, f"{metric}_file"].values[0]

results["MAE"]  = results["Model"].map(lambda m: rep(m, "MAE"))
results["RMSE"] = results["Model"].map(lambda m: rep(m, "RMSE"))
results["NASA"] = results["Model"].map(lambda m: rep(m, "NASA"))
results["MAE_std"] = results["Model"].map(
    lambda m: seed_info[m]["MAE_std"] if m in seed_info else np.nan)

# --- Table ---
print("=" * 78)
print("FINAL COMPARISON — every model sees each engine's full observed history")
print("  Tabular: 65-feature summary | Sequence: full observed sequence")
print("  Stochastic models: mean over five seeds")
print("=" * 78)
print(f"{'Model':<12}{'Input':<11}{'MAE':>8}{'±std':>8}{'RMSE':>8}{'NASA':>10}")
print("-" * 78)
for _, r in results.iterrows():
    std_txt = f"{r['MAE_std']:.2f}" if not np.isnan(r["MAE_std"]) else "det."
    print(f"{r['Model']:<12}{r['Input']:<11}{r['MAE']:>8.2f}{std_txt:>8}"
          f"{r['RMSE']:>8.2f}{r['NASA']:>10.2f}")
print("-" * 78)

results.to_csv(os.path.join(RESULTS_DIR, "fd001_final_comparison.csv"), index=False)

# --- Figure: MAE with seed variation, and NASA score. No titles; the
#     report supplies the caption. ---
order = ["LR", "XGBoost", "LSTM", "Chronos-2"]
results = results.set_index("Model").loc[order].reset_index()
colours = ["#999999", "#2E75B6", "#C55A11", "#548235"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))

err = results["MAE_std"].fillna(0).values
bars = ax1.bar(results["Model"], results["MAE"], yerr=err, capsize=5,
               color=colours, edgecolor="black", linewidth=1.1)
for b, v, inp in zip(bars, results["MAE"], results["Input"]):
    ax1.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                 textcoords="offset points", xytext=(0, 9),
                 ha="center", fontweight="bold", fontsize=9)
    ax1.annotate(f"({inp})", (b.get_x() + b.get_width()/2, v * 0.06),
                 textcoords="offset points", xytext=(0, 0), ha="center",
                 fontsize=8, color="white", fontweight="bold")
ax1.set_ylabel("MAE (cycles)")
ax1.grid(True, axis="y", alpha=0.3); ax1.set_axisbelow(True)

bars2 = ax2.bar(results["Model"], results["NASA"], color=colours,
                edgecolor="black", linewidth=1.1)
for b, v in zip(bars2, results["NASA"]):
    ax2.annotate(f"{v:.1f}", (b.get_x() + b.get_width()/2, v),
                 textcoords="offset points", xytext=(0, 6),
                 ha="center", fontweight="bold", fontsize=9)
ax2.set_ylabel("PHM score")
ax2.grid(True, axis="y", alpha=0.3); ax2.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_final_comparison.png"),
            dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: fd001_final_comparison.csv + fd001_final_comparison.png")